In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()

print("Current working directory:")
print(PROJECT_DIR)

assert PROJECT_DIR.name == "Enhanced", (
    "The notebook must be located and opened from the Enhanced folder."
)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("\nEnhancement Three workspace confirmed.")

Current working directory:
/Users/moniquehenry/Desktop/CS-499Capstone/CS340-GraziosoDashboard-Working/Enhanced

Enhancement Three workspace confirmed.


In [2]:
from config import AppConfig
from animal_shelter import AnimalShelter

config = AppConfig.from_env()

source_shelter = AnimalShelter(
    username=config.mongo_username,
    password=config.mongo_password,
    host=config.mongo_host,
    port=config.mongo_port,
    db=config.mongo_db,
    col=config.mongo_collection,
)

print("\nConnection information")
print("----------------------")
print("Connected:", source_shelter.ping())
print("Database:", source_shelter.database_name)
print("Collection:", source_shelter.collection_name)
print(
    "Record count:",
    source_shelter.collection.count_documents({}),
)

Connected to MongoDB successfully: aac.animals

Connection information
----------------------
Connected: True
Database: aac
Collection: animals
Record count: 10000


In [3]:
source_collection = source_shelter.collection

source_collection_name = source_collection.name

print("Source collection:", source_collection_name)
print(
    "Source record count:",
    source_collection.count_documents({}),
)

assert source_collection_name == "animals"

print(
    "\nThe original animals collection will be treated as read-only "
    "during migration development."
)

Source collection: animals
Source record count: 10000

The original animals collection will be treated as read-only during migration development.


In [4]:
duplicate_pipeline = [
    {
        "$match": {
            "animal_id": {
                "$exists": True,
                "$nin": [
                    None,
                    "",
                ],
            }
        }
    },
    {
        "$group": {
            "_id": "$animal_id",
            "record_count": {
                "$sum": 1,
            },
        }
    },
    {
        "$match": {
            "record_count": {
                "$gt": 1,
            }
        }
    },
    {
        "$count": "duplicate_animal_id_values",
    },
]

duplicate_results = list(
    source_collection.aggregate(
        duplicate_pipeline
    )
)

duplicate_animal_id_values = (
    duplicate_results[0][
        "duplicate_animal_id_values"
    ]
    if duplicate_results
    else 0
)

database_audit = {
    "total_records": (
        source_collection.count_documents({})
    ),

    "dog_records": (
        source_collection.count_documents(
            {
                "animal_type": "Dog",
            }
        )
    ),

    "distinct_animal_ids": len(
        source_collection.distinct(
            "animal_id"
        )
    ),

    "duplicate_animal_id_values": (
        duplicate_animal_id_values
    ),

    "missing_animal_ids": (
        source_collection.count_documents(
            {
                "$or": [
                    {
                        "animal_id": {
                            "$exists": False,
                        }
                    },
                    {
                        "animal_id": None,
                    },
                    {
                        "animal_id": "",
                    },
                ]
            }
        )
    ),

    "missing_names": (
        source_collection.count_documents(
            {
                "$or": [
                    {
                        "name": {
                            "$exists": False,
                        }
                    },
                    {
                        "name": None,
                    },
                    {
                        "name": "",
                    },
                ]
            }
        )
    ),

    "missing_outcome_types": (
        source_collection.count_documents(
            {
                "$or": [
                    {
                        "outcome_type": {
                            "$exists": False,
                        }
                    },
                    {
                        "outcome_type": None,
                    },
                    {
                        "outcome_type": "",
                    },
                ]
            }
        )
    ),

    "missing_age_values": (
        source_collection.count_documents(
            {
                "$or": [
                    {
                        "age_upon_outcome_in_weeks": {
                            "$exists": False,
                        }
                    },
                    {
                        "age_upon_outcome_in_weeks": None,
                    },
                ]
            }
        )
    ),

    "negative_age_values": (
        source_collection.count_documents(
            {
                "age_upon_outcome_in_weeks": {
                    "$lt": 0,
                }
            }
        )
    ),

    "missing_latitude": (
        source_collection.count_documents(
            {
                "$or": [
                    {
                        "location_lat": {
                            "$exists": False,
                        }
                    },
                    {
                        "location_lat": None,
                    },
                ]
            }
        )
    ),

    "missing_longitude": (
        source_collection.count_documents(
            {
                "$or": [
                    {
                        "location_long": {
                            "$exists": False,
                        }
                    },
                    {
                        "location_long": None,
                    },
                ]
            }
        )
    ),
}

database_audit

{'total_records': 10000,
 'dog_records': 5589,
 'distinct_animal_ids': 9860,
 'duplicate_animal_id_values': 138,
 'missing_animal_ids': 0,
 'missing_names': 3044,
 'missing_outcome_types': 2,
 'missing_age_values': 0,
 'negative_age_values': 1,
 'missing_latitude': 0,
 'missing_longitude': 0}

In [5]:
import pandas as pd

audit_table = pd.DataFrame(
    [
        {
            "Audit Measure": measure.replace(
                "_",
                " ",
            ).title(),
            "Count": count,
        }
        for measure, count
        in database_audit.items()
    ]
)

audit_table

,Audit Measure,Count
0,Total Records,10000
1,Dog Records,5589
2,Distinct Animal Ids,9860
3,Duplicate Animal Id Values,138
4,Missing Animal Ids,0
5,Missing Names,3044
6,Missing Outcome Types,2
7,Missing Age Values,0
8,Negative Age Values,1
9,Missing Latitude,0


In [6]:
import json

audit_path = (
    PROJECT_DIR
    / "docs"
    / "ENHANCEMENT_THREE_DATABASE_AUDIT.json"
)

audit_path.write_text(
    json.dumps(
        database_audit,
        indent=2,
    ),
    encoding="utf-8",
)

print("Database audit saved to:")
print(audit_path)

Database audit saved to:
/Users/moniquehenry/Desktop/CS-499Capstone/CS340-GraziosoDashboard-Working/Enhanced/docs/ENHANCEMENT_THREE_DATABASE_AUDIT.json


In [7]:
sample_negative_ages = list(
    source_collection.find(
        {
            "age_upon_outcome_in_weeks": {
                "$lt": 0,
            }
        },
        {
            "_id": 0,
            "animal_id": 1,
            "name": 1,
            "animal_type": 1,
            "breed": 1,
            "age_upon_outcome_in_weeks": 1,
            "outcome_type": 1,
        },
    ).limit(5)
)

sample_missing_outcomes = list(
    source_collection.find(
        {
            "$or": [
                {
                    "outcome_type": {
                        "$exists": False,
                    }
                },
                {
                    "outcome_type": None,
                },
                {
                    "outcome_type": "",
                },
            ]
        },
        {
            "_id": 0,
            "animal_id": 1,
            "name": 1,
            "animal_type": 1,
            "breed": 1,
            "age_upon_outcome_in_weeks": 1,
            "outcome_type": 1,
        },
    ).limit(5)
)

print("Sample negative-age records:")
display(
    pd.DataFrame(
        sample_negative_ages
    )
)

print("\nSample missing-outcome records:")
display(
    pd.DataFrame(
        sample_missing_outcomes
    )
)

Sample negative-age records:


,animal_id,animal_type,breed,name,outcome_type,age_upon_outcome_in_weeks
0,A749253,Cat,Domestic Shorthair Mix,Orange,Euthanasia,-7.043353



Sample missing-outcome records:


,animal_id,animal_type,breed,name,outcome_type,age_upon_outcome_in_weeks
0,A686025,Other,Bat Mix,,,52.336806
1,A744013,Other,Bat Mix,,,52.767857


In [9]:
import importlib
import database_setup

# Reload the module in case the file was edited while
# this notebook kernel was already running.
importlib.reload(database_setup)

from database_setup import (
    AUDIT_COLLECTION,
    ENHANCED_COLLECTION,
    setup_database,
)

database = source_shelter.database

setup_results = setup_database(
    database
)

setup_results

{'animal_collection': 'animals_enhanced',
 'animal_collection_status': 'updated',
 'animal_uid_index': 'uidx_record_uid',
 'audit_collection': 'audit_logs',
 'audit_collection_status': 'updated',
 'audit_timestamp_index': 'idx_audit_timestamp'}

In [10]:
collection_names = sorted(
    database.list_collection_names()
)

print("Available collections:")

for collection_name in collection_names:
    print("-", collection_name)

print(
    "\nOriginal animals count:",
    database["animals"].count_documents({}),
)

print(
    "Enhanced animals count:",
    database[
        ENHANCED_COLLECTION
    ].count_documents({}),
)

print(
    "Audit log count:",
    database[
        AUDIT_COLLECTION
    ].count_documents({}),
)

Available collections:
- animals
- animals_enhanced
- audit_logs

Original animals count: 10000
Enhanced animals count: 0
Audit log count: 0


In [11]:
animal_options = database[
    ENHANCED_COLLECTION
].options()

audit_options = database[
    AUDIT_COLLECTION
].options()

print("Animal validation level:")
print(
    animal_options.get(
        "validationLevel"
    )
)

print("\nAnimal validation action:")
print(
    animal_options.get(
        "validationAction"
    )
)

print("\nAnimal validator exists:")
print(
    "validator" in animal_options
)

print("\nAudit validator exists:")
print(
    "validator" in audit_options
)

Animal validation level:
strict

Animal validation action:
error

Animal validator exists:
True

Audit validator exists:
True


In [12]:
from datetime import datetime, timezone

from pymongo.errors import WriteError


enhanced_collection = database[
    ENHANCED_COLLECTION
]

invalid_test_uid = (
    "__enhancement_three_invalid_schema_test__"
)

invalid_document = {
    "record_uid": invalid_test_uid,
    "animal_id": "SCHEMA-TEST-INVALID",
    "animal_type": "Dog",
    "breed": "Test Breed",

    # This should fail because the schema requires
    # age_in_weeks to be zero or greater.
    "age_in_weeks": -5.0,

    "source_collection": "schema_test",
    "source_record_id": "invalid-test-record",
    "migrated_at": datetime.now(
        timezone.utc
    ),
}

try:
    enhanced_collection.insert_one(
        invalid_document
    )

except WriteError as error:
    print(
        "PASS: MongoDB rejected the invalid record."
    )

    print(
        "Error type:",
        type(error).__name__,
    )

else:
    # Safety cleanup if validation unexpectedly does not reject it.
    enhanced_collection.delete_one(
        {
            "record_uid": invalid_test_uid,
        }
    )

    raise AssertionError(
        "Schema validation did not reject the "
        "negative age value."
    )

PASS: MongoDB rejected the invalid record.
Error type: WriteError


In [13]:
valid_test_uid = (
    "__enhancement_three_valid_schema_test__"
)

valid_document = {
    "record_uid": valid_test_uid,
    "animal_id": "SCHEMA-TEST-VALID",
    "name": "Test Animal",
    "animal_type": "Dog",
    "breed": "Test Breed",
    "color": "Black",
    "sex_upon_outcome": "Neutered Male",
    "age_in_weeks": 52.0,
    "outcome_type": "Transfer",
    "outcome_subtype": None,
    "outcome_date": None,
    "date_of_birth": None,
    "location_lat": 30.2672,
    "location_long": -97.7431,
    "source_collection": "schema_test",
    "source_record_id": "valid-test-record",
    "migrated_at": datetime.now(
        timezone.utc
    ),
}

insert_result = enhanced_collection.insert_one(
    valid_document
)

print(
    "Valid record inserted:",
    insert_result.acknowledged,
)

saved_test_record = (
    enhanced_collection.find_one(
        {
            "record_uid": valid_test_uid,
        },
        {
            "_id": 0,
            "record_uid": 1,
            "animal_id": 1,
            "age_in_weeks": 1,
        },
    )
)

print(
    "Saved test record:",
    saved_test_record,
)

cleanup_result = (
    enhanced_collection.delete_one(
        {
            "record_uid": valid_test_uid,
        }
    )
)

print(
    "Test record removed:",
    cleanup_result.deleted_count == 1,
)

print(
    "Enhanced collection count:",
    enhanced_collection.count_documents({}),
)

Valid record inserted: True
Saved test record: {'record_uid': '__enhancement_three_valid_schema_test__', 'animal_id': 'SCHEMA-TEST-VALID', 'age_in_weeks': 52.0}
Test record removed: True
Enhanced collection count: 0


In [14]:
animal_indexes = list(
    enhanced_collection.list_indexes()
)

audit_indexes = list(
    database[
        AUDIT_COLLECTION
    ].list_indexes()
)

print("animals_enhanced indexes:")

for index in animal_indexes:
    print(
        "-",
        index["name"],
        dict(index["key"]),
        "unique:",
        index.get("unique", False),
    )

print("\naudit_logs indexes:")

for index in audit_indexes:
    print(
        "-",
        index["name"],
        dict(index["key"]),
    )

animals_enhanced indexes:
- _id_ {'_id': 1} unique: False
- uidx_record_uid {'record_uid': 1} unique: True

audit_logs indexes:
- _id_ {'_id': 1}
- idx_audit_timestamp {'timestamp': -1}


In [15]:
import json


database_setup_evidence = {
    "source_collection": "animals",
    "source_record_count": (
        database[
            "animals"
        ].count_documents({})
    ),
    "enhanced_collection": (
        ENHANCED_COLLECTION
    ),
    "enhanced_record_count": (
        enhanced_collection.count_documents({})
    ),
    "audit_collection": (
        AUDIT_COLLECTION
    ),
    "animal_validation_level": (
        animal_options.get(
            "validationLevel"
        )
    ),
    "animal_validation_action": (
        animal_options.get(
            "validationAction"
        )
    ),
    "animal_indexes": [
        index["name"]
        for index in animal_indexes
    ],
    "audit_indexes": [
        index["name"]
        for index in audit_indexes
    ],
    "invalid_record_rejected": True,
    "valid_record_accepted": True,
    "valid_test_record_removed": True,
}

setup_evidence_path = (
    PROJECT_DIR
    / "docs"
    / "ENHANCEMENT_THREE_DATABASE_SETUP.json"
)

setup_evidence_path.write_text(
    json.dumps(
        database_setup_evidence,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Database setup evidence saved to:"
)

print(
    setup_evidence_path
)

Database setup evidence saved to:
/Users/moniquehenry/Desktop/CS-499Capstone/CS340-GraziosoDashboard-Working/Enhanced/docs/ENHANCEMENT_THREE_DATABASE_SETUP.json


In [16]:
import importlib
import database_migration

importlib.reload(
    database_migration
)

from database_migration import (
    normalize_source_record,
)

preview_source_records = list(
    source_collection.find({})
    .limit(5)
)

preview_normalized_records = [
    normalize_source_record(
        source_record
    )
    for source_record
    in preview_source_records
]

preview_table = pd.DataFrame(
    preview_normalized_records
)

preview_table[
    [
        "record_uid",
        "animal_id",
        "name",
        "animal_type",
        "breed",
        "age_in_weeks",
        "outcome_type",
        "source_record_id",
    ]
]

,record_uid,animal_id,name,animal_type,breed,age_in_weeks,outcome_type,source_record_id
0,animals:6a516e87de234e6e36f9dde6,A733653,Kitty,Cat,Siamese Mix,30.822520,Adoption,6a516e87de234e6e36f9dde6
1,animals:6a516e87de234e6e36f9dde7,A700408,Nyla,Cat,Domestic Shorthair Mix,52.509325,Return to Owner,6a516e87de234e6e36f9dde7
2,animals:6a516e87de234e6e36f9dde8,A664843,Sherlock,Dog,Pit Bull Mix,62.246429,Transfer,6a516e87de234e6e36f9dde8
3,animals:6a516e87de234e6e36f9dde9,A691584,Luke,Dog,Labrador Retriever Mix,133.653571,Return to Owner,6a516e87de234e6e36f9dde9
4,animals:6a516e87de234e6e36f9ddea,A742287,*Kawhi,Dog,Boxer/Bullmastiff,107.931548,Adoption,6a516e87de234e6e36f9ddea


In [17]:
enhanced_collection = database[
    ENHANCED_COLLECTION
]

target_count_before_migration = (
    enhanced_collection.count_documents({})
)

print(
    "Enhanced records before migration:",
    target_count_before_migration,
)

assert target_count_before_migration == 0, (
    "The enhanced collection should be empty "
    "before the first migration."
)

Enhanced records before migration: 0


In [18]:
from database_migration import (
    migrate_animals,
)

migration_results = migrate_animals(
    database,
    batch_size=500,
)

migration_results

{'source_collection': 'animals',
 'target_collection': 'animals_enhanced',
 'source_count_before': 10000,
 'records_processed': 10000,
 'valid_records': 10000,
 'skipped_records': 0,
 'negative_or_invalid_ages_normalized': 1,
 'missing_names_preserved_as_null': 3044,
 'missing_outcomes_preserved_as_null': 2,
 'matched_records': 0,
 'modified_records': 0,
 'upserted_records': 10000,
 'migration_errors': [],
 'source_count_after': 10000,
 'target_count_after': 10000,
 'success': True}

In [19]:
source_count_after_migration = (
    database[
        "animals"
    ].count_documents({})
)

enhanced_count_after_migration = (
    database[
        ENHANCED_COLLECTION
    ].count_documents({})
)

print(
    "Original source records:",
    source_count_after_migration,
)

print(
    "Enhanced migrated records:",
    enhanced_count_after_migration,
)

assert (
    source_count_after_migration
    == migration_results[
        "source_count_before"
    ]
), (
    "The source collection count changed."
)

assert (
    enhanced_count_after_migration
    == migration_results[
        "valid_records"
    ]
), (
    "The enhanced collection count does not "
    "match the number of valid migrated records."
)

print(
    "\nPASS: Original records were preserved "
    "and normalized copies were migrated."
)

Original source records: 10000
Enhanced migrated records: 10000

PASS: Original records were preserved and normalized copies were migrated.


In [20]:
migration_validation = {
    "enhanced_record_count": (
        enhanced_collection.count_documents({})
    ),

    "distinct_record_uids": len(
        enhanced_collection.distinct(
            "record_uid"
        )
    ),

    "negative_age_values": (
        enhanced_collection.count_documents(
            {
                "age_in_weeks": {
                    "$lt": 0,
                }
            }
        )
    ),

    "missing_record_uids": (
        enhanced_collection.count_documents(
            {
                "$or": [
                    {
                        "record_uid": {
                            "$exists": False,
                        }
                    },
                    {
                        "record_uid": None,
                    },
                    {
                        "record_uid": "",
                    },
                ]
            }
        )
    ),

    "missing_animal_ids": (
        enhanced_collection.count_documents(
            {
                "$or": [
                    {
                        "animal_id": {
                            "$exists": False,
                        }
                    },
                    {
                        "animal_id": None,
                    },
                    {
                        "animal_id": "",
                    },
                ]
            }
        )
    ),

    "missing_breeds": (
        enhanced_collection.count_documents(
            {
                "$or": [
                    {
                        "breed": {
                            "$exists": False,
                        }
                    },
                    {
                        "breed": None,
                    },
                    {
                        "breed": "",
                    },
                ]
            }
        )
    ),
}

migration_validation

{'enhanced_record_count': 10000,
 'distinct_record_uids': 10000,
 'negative_age_values': 0,
 'missing_record_uids': 0,
 'missing_animal_ids': 0,
 'missing_breeds': 0}

In [21]:
assert (
    migration_validation[
        "enhanced_record_count"
    ]
    == migration_validation[
        "distinct_record_uids"
    ]
)

assert (
    migration_validation[
        "negative_age_values"
    ]
    == 0
)

print(
    "PASS: Migrated record identifiers are unique "
    "and negative age values were removed."
)

PASS: Migrated record identifiers are unique and negative age values were removed.


In [22]:
sample_migrated_records = list(
    enhanced_collection.find(
        {},
        {
            "_id": 0,
            "record_uid": 1,
            "animal_id": 1,
            "name": 1,
            "animal_type": 1,
            "breed": 1,
            "sex_upon_outcome": 1,
            "age_in_weeks": 1,
            "outcome_type": 1,
            "outcome_date": 1,
            "source_record_id": 1,
        },
    ).limit(10)
)

pd.DataFrame(
    sample_migrated_records
)

,record_uid,animal_id,name,animal_type,breed,sex_upon_outcome,age_in_weeks,outcome_type,outcome_date,source_record_id
0,animals:6a516e87de234e6e36f9dde6,A733653,Kitty,Cat,Siamese Mix,Intact Female,30.822520,Adoption,2016-08-27 18:11:00,6a516e87de234e6e36f9dde6
1,animals:6a516e87de234e6e36f9dde7,A700408,Nyla,Cat,Domestic Shorthair Mix,Spayed Female,52.509325,Return to Owner,2015-04-15 13:34:00,6a516e87de234e6e36f9dde7
2,animals:6a516e87de234e6e36f9dde8,A664843,Sherlock,Dog,Pit Bull Mix,Neutered Male,62.246429,Transfer,2014-08-18 17:24:00,6a516e87de234e6e36f9dde8
3,animals:6a516e87de234e6e36f9dde9,A691584,Luke,Dog,Labrador Retriever Mix,Neutered Male,133.653571,Return to Owner,2015-05-30 13:48:00,6a516e87de234e6e36f9dde9
4,animals:6a516e87de234e6e36f9ddea,A742287,*Kawhi,Dog,Boxer/Bullmastiff,Neutered Male,107.931548,Adoption,2017-02-11 12:30:00,6a516e87de234e6e36f9ddea
5,animals:6a516e87de234e6e36f9ddeb,A712638,Marcus,Dog,Pit Bull Mix,Neutered Male,198.820635,Transfer,2016-07-18 17:52:00,6a516e87de234e6e36f9ddeb
6,animals:6a516e87de234e6e36f9ddec,A723742,Gretchen,Dog,Miniature Schnauzer Mix,Spayed Female,261.818155,Adoption,2016-04-10 17:27:00,6a516e87de234e6e36f9ddec
7,animals:6a516e87de234e6e36f9dded,A668960,*Gigi,Dog,Pit Bull Mix,Spayed Female,28.386508,Adoption,2013-12-27 16:56:00,6a516e87de234e6e36f9dded
8,animals:6a516e87de234e6e36f9ddee,A721199,Belle,Dog,Dachshund Wirehair Mix,Spayed Female,52.820337,Adoption,2016-02-27 17:49:00,6a516e87de234e6e36f9ddee
9,animals:6a516e87de234e6e36f9ddef,A693288,NaN,Cat,Domestic Shorthair Mix,Spayed Female,10.396429,Adoption,2014-12-09 18:36:00,6a516e87de234e6e36f9ddef


In [23]:
latest_migration_audit = (
    database[
        AUDIT_COLLECTION
    ].find_one(
        {
            "action": "migration",
        },
        {
            "_id": 0,
        },
        sort=[
            (
                "timestamp",
                -1,
            )
        ],
    )
)

latest_migration_audit

{'record_uid': None,
 'source_record_id': None,
 'action': 'migration',
 'timestamp': datetime.datetime(2026, 7, 31, 17, 20, 2, 762000),
 'changed_fields': ['record_uid',
  'age_in_weeks',
  'outcome_date',
  'date_of_birth',
  'location_lat',
  'location_long'],
 'performed_by': 'database_migration.py',
 'success': True,
 'error_message': None,
 'details': {'source_count': 10000,
  'target_count': 10000,
  'valid_records': 10000,
  'skipped_records': 0}}

In [24]:
import json


migration_evidence = {
    "migration_results": migration_results,
    "migration_validation": (
        migration_validation
    ),
    "latest_audit_entry": {
        key: (
            value.isoformat()
            if isinstance(
                value,
                datetime,
            )
            else value
        )
        for key, value
        in latest_migration_audit.items()
    },
}

migration_evidence_path = (
    PROJECT_DIR
    / "docs"
    / "ENHANCEMENT_THREE_MIGRATION_RESULTS.json"
)

migration_evidence_path.write_text(
    json.dumps(
        migration_evidence,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print(
    "Migration evidence saved to:"
)

print(
    migration_evidence_path
)

Migration evidence saved to:
/Users/moniquehenry/Desktop/CS-499Capstone/CS340-GraziosoDashboard-Working/Enhanced/docs/ENHANCEMENT_THREE_MIGRATION_RESULTS.json


In [25]:
enhanced_record_count = (
    enhanced_collection.count_documents({})
)

source_record_count = (
    database["animals"].count_documents({})
)

print("Source records:", source_record_count)
print("Enhanced records:", enhanced_record_count)

assert enhanced_record_count > 0, (
    "Run the data migration before testing indexes."
)

print("\nMigration data is ready for index testing.")

Source records: 10000
Enhanced records: 10000

Migration data is ready for index testing.


In [26]:
indexes_before = list(
    enhanced_collection.list_indexes()
)

print("Indexes before query optimization:")

for index in indexes_before:
    print(
        "-",
        index["name"],
        dict(index["key"]),
        "unique:",
        index.get("unique", False),
    )

Indexes before query optimization:
- _id_ {'_id': 1} unique: False
- uidx_record_uid {'record_uid': 1} unique: True


In [27]:
common_breed_results = list(
    enhanced_collection.aggregate(
        [
            {
                "$match": {
                    "animal_type": "Dog",
                    "breed": {
                        "$nin": [
                            None,
                            "",
                        ]
                    },
                    "age_in_weeks": {
                        "$ne": None,
                    },
                }
            },
            {
                "$group": {
                    "_id": "$breed",
                    "record_count": {
                        "$sum": 1,
                    },
                }
            },
            {
                "$sort": {
                    "record_count": -1,
                    "_id": 1,
                }
            },
            {
                "$limit": 1,
            },
        ]
    )
)

assert common_breed_results, (
    "No valid dog breeds were found."
)

selected_test_breed = (
    common_breed_results[0]["_id"]
)

selected_test_breed_count = (
    common_breed_results[0]["record_count"]
)

print("Selected test breed:", selected_test_breed)
print(
    "Dog records for this breed:",
    selected_test_breed_count,
)

Selected test breed: Pit Bull Mix
Dog records for this breed: 801


In [28]:
breed_age_query = {
    "animal_type": "Dog",
    "breed": selected_test_breed,
    "age_in_weeks": {
        "$gte": 26.0,
        "$lte": 156.0,
    },
}

outcome_age_query = {
    "animal_type": "Dog",
    "outcome_type": "Transfer",
    "age_in_weeks": {
        "$gte": 20.0,
        "$lte": 300.0,
    },
}

query_projection = {
    "_id": 0,
    "record_uid": 1,
    "animal_id": 1,
    "name": 1,
    "breed": 1,
    "age_in_weeks": 1,
    "outcome_type": 1,
}

print(
    "Breed-age matches:",
    enhanced_collection.count_documents(
        breed_age_query
    ),
)

print(
    "Outcome-age matches:",
    enhanced_collection.count_documents(
        outcome_age_query
    ),
)

Breed-age matches: 467
Outcome-age matches: 748


In [29]:
from typing import Any


def run_find_explain(
    collection,
    query: dict[str, Any],
    projection: dict[str, int],
    hint: str | None = None,
) -> dict[str, Any]:
    """
    Run a MongoDB find command with execution statistics.

    An optional index hint can be supplied when verifying a
    specific index after it is created.
    """

    find_command: dict[str, Any] = {
        "find": collection.name,
        "filter": query,
        "projection": projection,
    }

    if hint is not None:
        find_command["hint"] = hint

    return collection.database.command(
        {
            "explain": find_command,
            "verbosity": "executionStats",
        }
    )


def collect_plan_stages(
    value: Any,
) -> set[str]:
    """Recursively collect stage names from an explain result."""

    stages: set[str] = set()

    if isinstance(value, dict):
        stage = value.get("stage")

        if isinstance(stage, str):
            stages.add(stage)

        for nested_value in value.values():
            stages.update(
                collect_plan_stages(
                    nested_value
                )
            )

    elif isinstance(value, list):
        for item in value:
            stages.update(
                collect_plan_stages(
                    item
                )
            )

    return stages


def summarize_explain(
    explain_result: dict[str, Any],
) -> dict[str, Any]:
    """Extract the primary query-performance measurements."""

    execution_stats = explain_result.get(
        "executionStats",
        {},
    )

    query_planner = explain_result.get(
        "queryPlanner",
        {},
    )

    winning_plan = query_planner.get(
        "winningPlan",
        {},
    )

    stages = sorted(
        collect_plan_stages(
            winning_plan
        )
    )

    return {
        "stages": stages,
        "n_returned": execution_stats.get(
            "nReturned"
        ),
        "documents_examined": (
            execution_stats.get(
                "totalDocsExamined"
            )
        ),
        "keys_examined": (
            execution_stats.get(
                "totalKeysExamined"
            )
        ),
        "execution_time_ms": (
            execution_stats.get(
                "executionTimeMillis"
            )
        ),
    }

In [30]:
breed_age_before_full = run_find_explain(
    enhanced_collection,
    breed_age_query,
    query_projection,
)

outcome_age_before_full = run_find_explain(
    enhanced_collection,
    outcome_age_query,
    query_projection,
)

breed_age_before = summarize_explain(
    breed_age_before_full
)

outcome_age_before = summarize_explain(
    outcome_age_before_full
)

print("Breed and age query — BEFORE indexes")
display(breed_age_before)

print("\nOutcome and age query — BEFORE indexes")
display(outcome_age_before)

Breed and age query — BEFORE indexes


{'stages': ['COLLSCAN', 'PROJECTION_SIMPLE'],
 'n_returned': 467,
 'documents_examined': 10000,
 'keys_examined': 0,
 'execution_time_ms': 18}


Outcome and age query — BEFORE indexes


{'stages': ['COLLSCAN', 'PROJECTION_SIMPLE'],
 'n_returned': 748,
 'documents_examined': 10000,
 'keys_examined': 0,
 'execution_time_ms': 10}

In [31]:
from bson import json_util


before_breed_path = (
    PROJECT_DIR
    / "docs"
    / "QUERY_PLAN_BEFORE_BREED_AGE.json"
)

before_outcome_path = (
    PROJECT_DIR
    / "docs"
    / "QUERY_PLAN_BEFORE_OUTCOME_AGE.json"
)

before_breed_path.write_text(
    json_util.dumps(
        breed_age_before_full,
        indent=2,
    ),
    encoding="utf-8",
)

before_outcome_path.write_text(
    json_util.dumps(
        outcome_age_before_full,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved:", before_breed_path.name)
print("Saved:", before_outcome_path.name)

Saved: QUERY_PLAN_BEFORE_BREED_AGE.json
Saved: QUERY_PLAN_BEFORE_OUTCOME_AGE.json


In [32]:
import importlib
import database_indexes

importlib.reload(
    database_indexes
)

from database_indexes import (
    ensure_query_indexes,
)

index_setup_results = ensure_query_indexes(
    enhanced_collection
)

index_setup_results

{'created_or_confirmed': ['idx_type_breed_age',
  'idx_type_outcome_age',
  'idx_animal_id',
  'idx_outcome_date'],
 'all_index_names': ['_id_',
  'uidx_record_uid',
  'idx_type_breed_age',
  'idx_type_outcome_age',
  'idx_animal_id',
  'idx_outcome_date']}

In [33]:
indexes_after = list(
    enhanced_collection.list_indexes()
)

index_table = pd.DataFrame(
    [
        {
            "Index Name": index["name"],
            "Fields": str(
                dict(index["key"])
            ),
            "Unique": index.get(
                "unique",
                False,
            ),
        }
        for index in indexes_after
    ]
)

index_table

,Index Name,Fields,Unique
0,_id_,{'_id': 1},False
1,uidx_record_uid,{'record_uid': 1},True
2,idx_type_breed_age,"{'animal_type': 1, 'breed': 1, 'age_in_weeks': 1}",False
3,idx_type_outcome_age,"{'animal_type': 1, 'outcome_type': 1, 'age_in_...",False
4,idx_animal_id,{'animal_id': 1},False
5,idx_outcome_date,{'outcome_date': -1},False


In [34]:
breed_age_after_full = run_find_explain(
    enhanced_collection,
    breed_age_query,
    query_projection,
)

outcome_age_after_full = run_find_explain(
    enhanced_collection,
    outcome_age_query,
    query_projection,
)

breed_age_after = summarize_explain(
    breed_age_after_full
)

outcome_age_after = summarize_explain(
    outcome_age_after_full
)

print("Breed and age query — AFTER indexes")
display(breed_age_after)

print("\nOutcome and age query — AFTER indexes")
display(outcome_age_after)

Breed and age query — AFTER indexes


{'stages': ['FETCH', 'IXSCAN', 'PROJECTION_SIMPLE'],
 'n_returned': 467,
 'documents_examined': 467,
 'keys_examined': 467,
 'execution_time_ms': 12}


Outcome and age query — AFTER indexes


{'stages': ['FETCH', 'IXSCAN', 'PROJECTION_SIMPLE'],
 'n_returned': 748,
 'documents_examined': 748,
 'keys_examined': 748,
 'execution_time_ms': 3}

In [39]:
def collect_index_names(
    value,
) -> set[str]:
    """Recursively collect index names from a plan document."""

    names: set[str] = set()

    if isinstance(value, dict):
        index_name = value.get("indexName")

        if isinstance(index_name, str):
            names.add(index_name)

        for nested_value in value.values():
            names.update(
                collect_index_names(
                    nested_value
                )
            )

    elif isinstance(value, list):
        for item in value:
            names.update(
                collect_index_names(
                    item
                )
            )

    return names


def collect_winning_index_names(
    explain_result,
) -> set[str]:
    """Return only indexes used by MongoDB's winning plan."""

    winning_plan = (
        explain_result
        .get("queryPlanner", {})
        .get("winningPlan", {})
    )

    return collect_index_names(
        winning_plan
    )


breed_indexes_used = collect_winning_index_names(
    breed_age_after_full
)

outcome_indexes_used = collect_winning_index_names(
    outcome_age_after_full
)

print(
    "Breed query winning index:",
    sorted(breed_indexes_used),
)

print(
    "Outcome query winning index:",
    sorted(outcome_indexes_used),
)

Breed query winning index: ['idx_type_breed_age']
Outcome query winning index: ['idx_type_outcome_age']


In [40]:
breed_age_hint_check = summarize_explain(
    run_find_explain(
        enhanced_collection,
        breed_age_query,
        query_projection,
        hint="idx_type_breed_age",
    )
)

outcome_age_hint_check = summarize_explain(
    run_find_explain(
        enhanced_collection,
        outcome_age_query,
        query_projection,
        hint="idx_type_outcome_age",
    )
)

print("Breed query with explicit index hint:")
display(breed_age_hint_check)

print("\nOutcome query with explicit index hint:")
display(outcome_age_hint_check)

Breed query with explicit index hint:


{'stages': ['FETCH', 'IXSCAN', 'PROJECTION_SIMPLE'],
 'n_returned': 467,
 'documents_examined': 467,
 'keys_examined': 467,
 'execution_time_ms': 2}


Outcome query with explicit index hint:


{'stages': ['FETCH', 'IXSCAN', 'PROJECTION_SIMPLE'],
 'n_returned': 748,
 'documents_examined': 748,
 'keys_examined': 748,
 'execution_time_ms': 2}

In [41]:
performance_comparison = pd.DataFrame(
    [
        {
            "Query": "Dog breed and age",
            "Phase": "Before indexes",
            "Stages": ", ".join(
                breed_age_before["stages"]
            ),
            "Returned": (
                breed_age_before[
                    "n_returned"
                ]
            ),
            "Documents Examined": (
                breed_age_before[
                    "documents_examined"
                ]
            ),
            "Keys Examined": (
                breed_age_before[
                    "keys_examined"
                ]
            ),
            "Execution Time (ms)": (
                breed_age_before[
                    "execution_time_ms"
                ]
            ),
        },
        {
            "Query": "Dog breed and age",
            "Phase": "After indexes",
            "Stages": ", ".join(
                breed_age_after["stages"]
            ),
            "Returned": (
                breed_age_after[
                    "n_returned"
                ]
            ),
            "Documents Examined": (
                breed_age_after[
                    "documents_examined"
                ]
            ),
            "Keys Examined": (
                breed_age_after[
                    "keys_examined"
                ]
            ),
            "Execution Time (ms)": (
                breed_age_after[
                    "execution_time_ms"
                ]
            ),
        },
        {
            "Query": "Dog outcome and age",
            "Phase": "Before indexes",
            "Stages": ", ".join(
                outcome_age_before[
                    "stages"
                ]
            ),
            "Returned": (
                outcome_age_before[
                    "n_returned"
                ]
            ),
            "Documents Examined": (
                outcome_age_before[
                    "documents_examined"
                ]
            ),
            "Keys Examined": (
                outcome_age_before[
                    "keys_examined"
                ]
            ),
            "Execution Time (ms)": (
                outcome_age_before[
                    "execution_time_ms"
                ]
            ),
        },
        {
            "Query": "Dog outcome and age",
            "Phase": "After indexes",
            "Stages": ", ".join(
                outcome_age_after[
                    "stages"
                ]
            ),
            "Returned": (
                outcome_age_after[
                    "n_returned"
                ]
            ),
            "Documents Examined": (
                outcome_age_after[
                    "documents_examined"
                ]
            ),
            "Keys Examined": (
                outcome_age_after[
                    "keys_examined"
                ]
            ),
            "Execution Time (ms)": (
                outcome_age_after[
                    "execution_time_ms"
                ]
            ),
        },
    ]
)

performance_comparison

,Query,Phase,Stages,Returned,Documents Examined,Keys Examined,Execution Time (ms)
0,Dog breed and age,Before indexes,"COLLSCAN, PROJECTION_SIMPLE",467,10000,0,18
1,Dog breed and age,After indexes,"FETCH, IXSCAN, PROJECTION_SIMPLE",467,467,467,12
2,Dog outcome and age,Before indexes,"COLLSCAN, PROJECTION_SIMPLE",748,10000,0,10
3,Dog outcome and age,After indexes,"FETCH, IXSCAN, PROJECTION_SIMPLE",748,748,748,3


In [42]:
assert (
    breed_age_before["n_returned"]
    == breed_age_after["n_returned"]
), (
    "The breed query returned a different number "
    "of records after indexing."
)

assert (
    outcome_age_before["n_returned"]
    == outcome_age_after["n_returned"]
), (
    "The outcome query returned a different number "
    "of records after indexing."
)

print(
    "PASS: Both indexed queries returned the same "
    "records as their unindexed versions."
)

PASS: Both indexed queries returned the same records as their unindexed versions.


In [43]:
from bson import json_util


after_breed_path = (
    PROJECT_DIR
    / "docs"
    / "QUERY_PLAN_AFTER_BREED_AGE.json"
)

after_outcome_path = (
    PROJECT_DIR
    / "docs"
    / "QUERY_PLAN_AFTER_OUTCOME_AGE.json"
)

comparison_path = (
    PROJECT_DIR
    / "docs"
    / "ENHANCEMENT_THREE_INDEX_COMPARISON.csv"
)

after_breed_path.write_text(
    json_util.dumps(
        breed_age_after_full,
        indent=2,
    ),
    encoding="utf-8",
)

after_outcome_path.write_text(
    json_util.dumps(
        outcome_age_after_full,
        indent=2,
    ),
    encoding="utf-8",
)

performance_comparison.to_csv(
    comparison_path,
    index=False,
)

print("Saved:", after_breed_path.name)
print("Saved:", after_outcome_path.name)
print("Saved:", comparison_path.name)

Saved: QUERY_PLAN_AFTER_BREED_AGE.json
Saved: QUERY_PLAN_AFTER_OUTCOME_AGE.json
Saved: ENHANCEMENT_THREE_INDEX_COMPARISON.csv


In [44]:
def format_measurement(
    value,
) -> str:
    """Return a readable explain-plan measurement."""

    if value is None:
        return "Not reported"

    return str(value)


performance_summary = f"""
ENHANCEMENT THREE DATABASE INDEX EVALUATION
===========================================

Collection:
{ENHANCED_COLLECTION}

Total enhanced records:
{enhanced_collection.count_documents({})}

QUERY ONE: DOG BREED AND AGE
----------------------------
Selected breed:
{selected_test_breed}

Before indexing:
Plan stages: {", ".join(breed_age_before["stages"])}
Records returned: {format_measurement(breed_age_before["n_returned"])}
Documents examined: {format_measurement(breed_age_before["documents_examined"])}
Keys examined: {format_measurement(breed_age_before["keys_examined"])}
Execution time: {format_measurement(breed_age_before["execution_time_ms"])} ms

After indexing:
Plan stages: {", ".join(breed_age_after["stages"])}
Records returned: {format_measurement(breed_age_after["n_returned"])}
Documents examined: {format_measurement(breed_age_after["documents_examined"])}
Keys examined: {format_measurement(breed_age_after["keys_examined"])}
Execution time: {format_measurement(breed_age_after["execution_time_ms"])} ms
Winning index: {", ".join(sorted(breed_indexes_used)) or "None"}

QUERY TWO: DOG OUTCOME AND AGE
------------------------------
Selected outcome:
Transfer

Before indexing:
Plan stages: {", ".join(outcome_age_before["stages"])}
Records returned: {format_measurement(outcome_age_before["n_returned"])}
Documents examined: {format_measurement(outcome_age_before["documents_examined"])}
Keys examined: {format_measurement(outcome_age_before["keys_examined"])}
Execution time: {format_measurement(outcome_age_before["execution_time_ms"])} ms

After indexing:
Plan stages: {", ".join(outcome_age_after["stages"])}
Records returned: {format_measurement(outcome_age_after["n_returned"])}
Documents examined: {format_measurement(outcome_age_after["documents_examined"])}
Keys examined: {format_measurement(outcome_age_after["keys_examined"])}
Execution time: {format_measurement(outcome_age_after["execution_time_ms"])} ms
Winning index: {", ".join(sorted(outcome_indexes_used)) or "None"}

INTERPRETATION
--------------
Both indexed queries returned the same logical records as their
unindexed versions.

The breed-and-age query reduced documents examined from
{breed_age_before["documents_examined"]} to
{breed_age_after["documents_examined"]}.

The outcome-and-age query reduced documents examined from
{outcome_age_before["documents_examined"]} to
{outcome_age_after["documents_examined"]}.

The compound indexes improve repeated dashboard searches but require
additional storage and index-maintenance work during create, update,
and delete operations. This trade-off is appropriate because the
dashboard performs frequent searches and relatively infrequent writes.
""".strip()

summary_path = (
    PROJECT_DIR
    / "docs"
    / "ENHANCEMENT_THREE_INDEX_SUMMARY.txt"
)

summary_path.write_text(
    performance_summary,
    encoding="utf-8",
)

print(performance_summary)
print("\nSaved:", summary_path)

ENHANCEMENT THREE DATABASE INDEX EVALUATION

Collection:
animals_enhanced

Total enhanced records:
10000

QUERY ONE: DOG BREED AND AGE
----------------------------
Selected breed:
Pit Bull Mix

Before indexing:
Plan stages: COLLSCAN, PROJECTION_SIMPLE
Records returned: 467
Documents examined: 10000
Keys examined: 0
Execution time: 18 ms

After indexing:
Plan stages: FETCH, IXSCAN, PROJECTION_SIMPLE
Records returned: 467
Documents examined: 467
Keys examined: 467
Execution time: 12 ms
Winning index: idx_type_breed_age

QUERY TWO: DOG OUTCOME AND AGE
------------------------------
Selected outcome:
Transfer

Before indexing:
Plan stages: COLLSCAN, PROJECTION_SIMPLE
Records returned: 748
Documents examined: 10000
Keys examined: 0
Execution time: 10 ms

After indexing:
Plan stages: FETCH, IXSCAN, PROJECTION_SIMPLE
Records returned: 748
Documents examined: 748
Keys examined: 748
Execution time: 3 ms
Winning index: idx_type_outcome_age

INTERPRETATION
--------------
Both indexed queries retu

In [45]:
import importlib
import animal_shelter

importlib.reload(
    animal_shelter
)

from animal_shelter import AnimalShelter

In [46]:
enhanced_shelter = AnimalShelter(
    username=config.mongo_username,
    password=config.mongo_password,
    host=config.mongo_host,
    port=config.mongo_port,
    db=config.mongo_db,

    # Use the normalized Enhancement Three collection.
    col="animals_enhanced",
)

print(
    "Enhanced collection:",
    enhanced_shelter.collection_name,
)

print(
    "Enhanced record count:",
    enhanced_shelter.collection.count_documents({}),
)

Connected to MongoDB successfully: aac.animals_enhanced
Enhanced collection: animals_enhanced
Enhanced record count: 10000


In [47]:
controlled_query = (
    enhanced_shelter.build_animal_query(
        animal_type="Dog",
        breed="Pit Bull Mix",
        outcome_type="Transfer",
        age_range=(
            26,
            156,
        ),
    )
)

controlled_query

{'animal_type': 'Dog',
 'breed': 'Pit Bull Mix',
 'outcome_type': 'Transfer',
 'age_in_weeks': {'$gte': 26.0, '$lte': 156.0}}

In [48]:
page_one = (
    enhanced_shelter.find_animals_page(
        animal_type="Dog",
        breed=selected_test_breed,
        age_range=(
            26,
            156,
        ),
        page=1,
        page_size=10,
        sort_field="animal_id",
    )
)

print(
    "Page:",
    page_one["page"],
)

print(
    "Page size:",
    page_one["page_size"],
)

print(
    "Records on page:",
    len(
        page_one["records"]
    ),
)

print(
    "Total matching records:",
    page_one["total_records"],
)

print(
    "Total pages:",
    page_one["total_pages"],
)

pd.DataFrame(
    page_one["records"]
).head()

Page: 1
Page size: 10
Records on page: 10
Total matching records: 467
Total pages: 47


,record_uid,animal_id,name,animal_type,breed,sex_upon_outcome,age_in_weeks,outcome_type,location_lat,location_long,age_upon_outcome_in_weeks
0,animals:6a516e88de234e6e36f9f4cc,A616640,*Joy,Dog,Pit Bull Mix,Spayed Female,153.094147,Return to Owner,30.344625,-97.269004,153.094147
1,animals:6a516e88de234e6e36f9f599,A620318,Guthor,Dog,Pit Bull Mix,Neutered Male,123.390972,Return to Owner,30.412780,-97.689075,123.390972
2,animals:6a516e87de234e6e36f9e19f,A624126,Blue,Dog,Pit Bull Mix,Neutered Male,94.502579,Return to Owner,30.425495,-97.559376,94.502579
3,animals:6a516e88de234e6e36f9f263,A637506,Molly,Dog,Pit Bull Mix,Spayed Female,128.253274,Euthanasia,30.676055,-97.318285,128.253274
4,animals:6a516e88de234e6e36f9e6a8,A637670,Shadow,Dog,Pit Bull Mix,Spayed Female,127.496032,Return to Owner,30.553246,-97.485278,127.496032


In [49]:
assert len(
    page_one["records"]
) == 10

assert (
    page_one["total_records"]
    == breed_age_after[
        "n_returned"
    ]
)

print(
    "PASS: Pagination returned the expected "
    "number of records."
)

PASS: Pagination returned the expected number of records.


In [50]:
first_projected_record = (
    page_one["records"][0]
)

print(
    "Returned fields:",
    sorted(
        first_projected_record.keys()
    ),
)

assert "_id" not in first_projected_record
assert "source_record_id" not in first_projected_record
assert "migrated_at" not in first_projected_record

assert (
    "age_upon_outcome_in_weeks"
    in first_projected_record
)

print(
    "PASS: Projection excluded internal and "
    "unnecessary database fields."
)

Returned fields: ['age_in_weeks', 'age_upon_outcome_in_weeks', 'animal_id', 'animal_type', 'breed', 'location_lat', 'location_long', 'name', 'outcome_type', 'record_uid', 'sex_upon_outcome']
PASS: Projection excluded internal and unnecessary database fields.


In [51]:
database_breeds = (
    enhanced_shelter.distinct_values(
        "breed",
        animal_type="Dog",
    )
)

database_outcomes = (
    enhanced_shelter.distinct_values(
        "outcome_type",
        animal_type="Dog",
    )
)

print(
    "Distinct dog breeds:",
    len(database_breeds),
)

print(
    "Distinct dog outcomes:",
    database_outcomes,
)

assert database_breeds
assert database_outcomes

print(
    "PASS: Distinct dashboard options were "
    "retrieved directly from MongoDB."
)

Distinct dog breeds: 711
Distinct dog outcomes: ['Adoption', 'Died', 'Disposal', 'Euthanasia', 'Missing', 'Return to Owner', 'Rto-Adopt', 'Transfer']
PASS: Distinct dashboard options were retrieved directly from MongoDB.


In [52]:
try:
    enhanced_shelter.distinct_values(
        "password"
    )

except ValueError as error:
    print(
        "PASS: Unsupported field rejected."
    )

    print(error)

else:
    raise AssertionError(
        "Unsupported distinct field was accepted."
    )

PASS: Unsupported field rejected.
Unsupported distinct field: password


In [53]:
dog_age_bounds = (
    enhanced_shelter.age_bounds(
        animal_type="Dog"
    )
)

outcome_statistics = (
    enhanced_shelter.outcome_summary(
        animal_type="Dog"
    )
)

print(
    "Dog age bounds:",
    dog_age_bounds,
)

display(
    pd.DataFrame(
        outcome_statistics
    )
)

assert dog_age_bounds is not None
assert outcome_statistics

print(
    "PASS: MongoDB aggregation returned "
    "dashboard statistics."
)

Dog age bounds: (0.0813492063492064, 940.239583333333)


,total,outcome_type,average_age
0,2539,Adoption,110.60
1,1644,Return to Owner,213.55
2,1180,Transfer,129.21
3,191,Euthanasia,249.96
4,16,Died,86.87
5,14,Rto-Adopt,139.35
6,3,Missing,72.92
7,2,Disposal,29.79


PASS: MongoDB aggregation returned dashboard statistics.


In [54]:
pagination_safeguards = {
    "page_zero_rejected": False,
    "oversized_page_rejected": False,
    "unsupported_sort_rejected": False,
}

try:
    enhanced_shelter.find_animals_page(
        page=0
    )

except ValueError:
    pagination_safeguards[
        "page_zero_rejected"
    ] = True


try:
    enhanced_shelter.find_animals_page(
        page_size=500
    )

except ValueError:
    pagination_safeguards[
        "oversized_page_rejected"
    ] = True


try:
    enhanced_shelter.find_animals_page(
        sort_field="unknown_field"
    )

except ValueError:
    pagination_safeguards[
        "unsupported_sort_rejected"
    ] = True


pagination_safeguards

{'page_zero_rejected': True,
 'oversized_page_rejected': True,
 'unsupported_sort_rejected': True}

In [55]:
read_layer_evidence = {
    "collection": (
        enhanced_shelter.collection_name
    ),
    "collection_count": (
        enhanced_shelter.collection.count_documents({})
    ),
    "test_breed": selected_test_breed,
    "page": page_one["page"],
    "page_size": page_one["page_size"],
    "records_on_page": len(
        page_one["records"]
    ),
    "total_matching_records": (
        page_one["total_records"]
    ),
    "total_pages": (
        page_one["total_pages"]
    ),
    "internal_id_excluded": (
        "_id"
        not in first_projected_record
    ),
    "distinct_breed_count": len(
        database_breeds
    ),
    "distinct_outcomes": (
        database_outcomes
    ),
    "dog_age_bounds": (
        dog_age_bounds
    ),
    "outcome_summary": (
        outcome_statistics
    ),
    "pagination_safeguards": (
        pagination_safeguards
    ),
}

read_layer_path = (
    PROJECT_DIR
    / "docs"
    / "ENHANCEMENT_THREE_READ_LAYER_RESULTS.json"
)

read_layer_path.write_text(
    json.dumps(
        read_layer_evidence,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print(
    "Read-layer evidence saved to:"
)

print(
    read_layer_path
)

Read-layer evidence saved to:
/Users/moniquehenry/Desktop/CS-499Capstone/CS340-GraziosoDashboard-Working/Enhanced/docs/ENHANCEMENT_THREE_READ_LAYER_RESULTS.json


In [60]:
import importlib
import animal_shelter

importlib.reload(
    animal_shelter
)

from animal_shelter import AnimalShelter


crud_shelter = AnimalShelter(
    username=config.mongo_username,
    password=config.mongo_password,
    host=config.mongo_host,
    port=config.mongo_port,
    db=config.mongo_db,
    col="animals_enhanced",
)

crud_baseline_count = (
    crud_shelter.collection.count_documents({})
)

print(
    "CRUD baseline count:",
    crud_baseline_count,
)

Connected to MongoDB successfully: aac.animals_enhanced
CRUD baseline count: 10000


In [61]:
crud_test_uid = None

try:
    created_record = (
        crud_shelter.create_animal_record(
            {
                "animal_id": "CS499-CRUD-TEST",
                "name": "Database Test Animal",
                "animal_type": "Dog",
                "breed": "Labrador Retriever Mix",
                "color": "Black",
                "sex_upon_outcome": "Neutered Male",
                "age_in_weeks": 104.0,
                "outcome_type": "Transfer",
                "outcome_subtype": None,
                "location_lat": 30.2672,
                "location_long": -97.7431,
            },
            performed_by=(
                "enhancement_three_notebook"
            ),
        )
    )

    crud_test_uid = (
        created_record[
            "record_uid"
        ]
    )

    print(
        "Created UID:",
        crud_test_uid,
    )

    saved_record = (
        crud_shelter.get_animal_by_uid(
            crud_test_uid
        )
    )

    assert saved_record is not None
    assert (
        saved_record["animal_id"]
        == "CS499-CRUD-TEST"
    )

    update_result = (
        crud_shelter.update_animal_record(
            crud_test_uid,
            {
                "name": (
                    "Updated Database Test Animal"
                ),
                "outcome_type": "Adoption",
                "age_in_weeks": 105.0,
            },
            performed_by=(
                "enhancement_three_notebook"
            ),
        )
    )

    assert update_result["matched"] == 1

    updated_record = (
        crud_shelter.get_animal_by_uid(
            crud_test_uid
        )
    )

    assert (
        updated_record["name"]
        == "Updated Database Test Animal"
    )

    assert (
        updated_record["outcome_type"]
        == "Adoption"
    )

    print(
        "Update result:",
        update_result,
    )

    try:
        crud_shelter.delete_animal_record(
            crud_test_uid,
            performed_by=(
                "enhancement_three_notebook"
            ),
        )

    except ValueError:
        print(
            "PASS: Delete without confirmation "
            "was rejected."
        )

    else:
        raise AssertionError(
            "Delete without confirmation "
            "was accepted."
        )

    deletion_result = (
        crud_shelter.delete_animal_record(
            crud_test_uid,
            confirm=True,
            performed_by=(
                "enhancement_three_notebook"
            ),
        )
    )

    assert deletion_result is True

    assert (
        crud_shelter.get_animal_by_uid(
            crud_test_uid
        )
        is None
    )

    crud_test_uid = None

    print(
        "PASS: Secure create, read, update, "
        "and delete operations succeeded."
    )

finally:
    # Emergency cleanup if an earlier assertion fails.
    if crud_test_uid is not None:
        crud_shelter.collection.delete_one(
            {
                "record_uid": crud_test_uid,
            }
        )

Created UID: application:5cfdbbbcaa1c4211a405a0c23f9fc88a
Update result: {'matched': 1, 'modified': 1}
PASS: Delete without confirmation was rejected.
PASS: Secure create, read, update, and delete operations succeeded.


In [63]:
# Recreate the source collection object after reloading
# the updated AnimalShelter class.

source_shelter = AnimalShelter(
    username=config.mongo_username,
    password=config.mongo_password,
    host=config.mongo_host,
    port=config.mongo_port,
    db=config.mongo_db,
    col=config.mongo_collection,
)

print(
    "Source collection:",
    source_shelter.collection_name,
)

print(
    "Secure create method available:",
    hasattr(
        source_shelter,
        "create_animal_record",
    ),
)

Connected to MongoDB successfully: aac.animals
Source collection: animals
Secure create method available: True


In [64]:
crud_safeguards = {
    "missing_required_field_rejected": False,
    "unsupported_field_rejected": False,
    "negative_age_rejected": False,
    "source_collection_protected": False,
}


try:
    crud_shelter.create_animal_record(
        {
            "animal_id": "INVALID-TEST",
            "animal_type": "Dog",
        },
        performed_by="validation_test",
    )

except ValueError:
    crud_safeguards[
        "missing_required_field_rejected"
    ] = True


try:
    crud_shelter.create_animal_record(
        {
            "animal_id": "INVALID-TEST",
            "animal_type": "Dog",
            "breed": "Test Breed",
            "$where": "malicious input",
        },
        performed_by="validation_test",
    )

except ValueError:
    crud_safeguards[
        "unsupported_field_rejected"
    ] = True


try:
    crud_shelter.create_animal_record(
        {
            "animal_id": "INVALID-TEST",
            "animal_type": "Dog",
            "breed": "Test Breed",
            "age_in_weeks": -1,
        },
        performed_by="validation_test",
    )

except ValueError:
    crud_safeguards[
        "negative_age_rejected"
    ] = True


try:
    source_shelter.create_animal_record(
        {
            "animal_id": "SOURCE-PROTECTION-TEST",
            "animal_type": "Dog",
            "breed": "Test Breed",
        },
        performed_by="validation_test",
    )

except RuntimeError:
    crud_safeguards[
        "source_collection_protected"
    ] = True


crud_safeguards

{'missing_required_field_rejected': True,
 'unsupported_field_rejected': True,
 'negative_age_rejected': True,
 'source_collection_protected': True}

In [65]:
audit_entries = list(
    crud_shelter.database[
        "audit_logs"
    ].find(
        {
            "record_uid": created_record[
                "record_uid"
            ],
        },
        {
            "_id": 0,
        },
    ).sort(
        "timestamp",
        1,
    )
)

audit_table = pd.DataFrame(
    audit_entries
)

audit_table[
    [
        "action",
        "success",
        "performed_by",
        "changed_fields",
        "error_message",
    ]
]

,action,success,performed_by,changed_fields,error_message
0,create,True,enhancement_three_notebook,"[age_in_weeks, animal_id, animal_type, breed, ...",NaN
1,update,True,enhancement_three_notebook,"[age_in_weeks, name, outcome_type]",NaN
2,delete,False,enhancement_three_notebook,[],Deletion confirmation was not supplied.
3,delete,True,enhancement_three_notebook,[],NaN


In [66]:
successful_actions = {
    entry["action"]
    for entry in audit_entries
    if entry["success"] is True
}

assert {
    "create",
    "update",
    "delete",
}.issubset(
    successful_actions
)

assert any(
    entry["action"] == "delete"
    and entry["success"] is False
    for entry in audit_entries
)

print(
    "PASS: Successful and rejected CRUD "
    "operations were recorded in audit_logs."
)

PASS: Successful and rejected CRUD operations were recorded in audit_logs.


In [67]:
crud_final_count = (
    crud_shelter.collection.count_documents({})
)

print(
    "CRUD baseline count:",
    crud_baseline_count,
)

print(
    "CRUD final count:",
    crud_final_count,
)

assert (
    crud_final_count
    == crud_baseline_count
)

print(
    "PASS: CRUD testing did not leave a "
    "temporary animal record."
)

CRUD baseline count: 10000
CRUD final count: 10000
PASS: CRUD testing did not leave a temporary animal record.


In [68]:
crud_evidence = {
    "collection": (
        crud_shelter.collection.name
    ),
    "baseline_count": crud_baseline_count,
    "final_count": crud_final_count,
    "temporary_record_removed": (
        crud_baseline_count
        == crud_final_count
    ),
    "created_record_uid": (
        created_record[
            "record_uid"
        ]
    ),
    "update_result": update_result,
    "delete_confirmed": deletion_result,
    "safeguards": crud_safeguards,
    "audit_entries": audit_entries,
}

crud_evidence_path = (
    PROJECT_DIR
    / "docs"
    / "ENHANCEMENT_THREE_SECURE_CRUD_RESULTS.json"
)

crud_evidence_path.write_text(
    json.dumps(
        crud_evidence,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print(
    "Secure CRUD evidence saved to:"
)

print(
    crud_evidence_path
)

Secure CRUD evidence saved to:
/Users/moniquehenry/Desktop/CS-499Capstone/CS340-GraziosoDashboard-Working/Enhanced/docs/ENHANCEMENT_THREE_SECURE_CRUD_RESULTS.json
